In [1]:
import sys
from pathlib import Path

p = Path.cwd()
while not (p / "src").exists() and p.parent != p:
    p = p.parent
sys.path.append(str(p))
RESULTS = p / "results"
RESULTS.mkdir(exist_ok=True)
print("Project root:", p)

Project root: C:\Users\Administrator\Documents\code\limit-order-book


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src.simulator import Simulator, SimulationConfig
from src.metrics import compute_market_stats, impact_curve

print("Imports OK")

Imports OK


In [3]:
cfg = SimulationConfig(seed=42, duration=10.0)
result = Simulator(cfg).run()
stats = compute_market_stats(result)
print(f"Trades:        {stats.n_trades}")
print(f"Volume:        {stats.total_volume:.0f}")
print(f"Mean trade:    {stats.mean_trade_size:.1f}")
print(f"Volatility:    {stats.volatility:.6f}")
print(f"Top depth B/A: {stats.mean_top_depth_bid:.0f} / {stats.mean_top_depth_ask:.0f}")

Trades:        485
Volume:        1511
Mean trade:    3.1
Volatility:    0.000062
Top depth B/A: 2 / 6


In [7]:
def factory(seed):
    cfg = SimulationConfig(
        seed=seed,
        duration=2.0,
        n_levels=10,
        initial_qty=5,
        order_rate=100.0,
        cancel_rate=50.0,
    )
    return Simulator(cfg)

sizes = [2, 5, 10, 20, 40, 80, 150, 250]
curve = impact_curve(factory, sizes, n_trials=10)

for s, i, e in zip(curve["sizes"], curve["impacts"], curve["errors"]):
    print(f"size={s:4d}  impact={i:+.4f} ± {e:.4f}")

size=   2  impact=+0.0445 ± 0.0026
size=   5  impact=+0.0423 ± 0.0009
size=  10  impact=+0.0465 ± 0.0028
size=  20  impact=+0.0509 ± 0.0026
size=  40  impact=+0.0643 ± 0.0037
size=  80  impact=+0.0768 ± 0.0061
size= 150  impact=+0.0890 ± 0.0077
size= 250  impact=+0.0953 ± 0.0051


In [8]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(curve["sizes"], curve["impacts"], yerr=curve["errors"],
            fmt="o-", capsize=4, markersize=6, label="simulated")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Order size (shares)")
ax.set_ylabel("Market impact (price units)")
ax.set_title("Market impact vs order size (log-log)")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "impact_curve.png", dpi=120)
plt.show()
print("Saved results/impact_curve.png")

Saved results/impact_curve.png


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4420\3969712625.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
log_sizes = np.log(curve["sizes"])
log_impacts = np.log(np.array(curve["impacts"]))
mask = np.isfinite(log_impacts) & (np.array(curve["impacts"]) > 0)
slope, intercept = np.polyfit(log_sizes[mask], log_impacts[mask], 1)
print(f"Fitted power-law exponent: {slope:.3f}")
print(f"Coefficient (exp of intercept): {np.exp(intercept):.4f}")

# Overlay fit
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(curve["sizes"], curve["impacts"], yerr=curve["errors"],
            fmt="o", capsize=4, markersize=6, label="simulated")
size_fit = np.array(curve["sizes"])
impact_fit = np.exp(intercept) * size_fit ** slope
ax.plot(size_fit, impact_fit, "--",
        label=f"fit: {slope:.2f}·size^{slope:.2f}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Order size (shares)")
ax.set_ylabel("Market impact (price units)")
ax.set_title(f"Market impact: power-law exponent = {slope:.2f}")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "impact_fit.png", dpi=120)
plt.show()
print("Saved results/impact_fit.png")

Fitted power-law exponent: 0.185
Coefficient (exp of intercept): 0.0332
Saved results/impact_fit.png


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4420\1458797533.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Market impact

**What this shows:** market impact grows **concavely** with order size —
larger orders cost more per share, but not linearly. This is the most
robust empirical regularity in market microstructure.

**Why it happens:** at each price level, resting liquidity must be
consumed. But the book gets denser as you move away from the touch
(more orders accumulate at worse prices), and other traders refresh
liquidity as you consume it. So marginal impact per share decreases.

**Theoretical benchmark:** the "square-root law" predicts exponent ≈ 0.5.

**What we observed:** the fitted exponent is printed above. A value in
[0.3, 0.7] reproduces the qualitative shape. Values near 1.0 indicate a
book too shallow for the order sizes tested; values near 0 indicate a
book too deep.

**How to make it more realistic:** use log-normal order sizes (heavy tail),
add transient impact (traders reverting), or use queue-reactive order flow
where arrival rates depend on book imbalance.